# Notebook 6a — UHCD Merge & Disposition Harmonisation

## Purpose

This notebook integrates the UHCD (Unité d'Hospitalisation de Courte Durée — short-stay observation unit) patient list and constructs a harmonised `disposition` outcome variable by reconciling three heterogeneous discharge-destination fields present in the dataset.

---

## Pipeline Overview

### 1. UHCD data loading and merge
The UHCD patient list (`nda_uhcd_2022_2023.csv`) is loaded and left-joined onto the master dataset, restricted to 2022–2023 admissions. A boolean `UHCD` flag is added to each patient row.

### 2. Audit of discharge-destination fields
Three fields encode patient destination at different levels:
- `decision_urgence` — physician decision at ED discharge (from the administrative file)
- `disposition_med` — discharge destination recorded in the medical file
- `UHCD` — boolean flag from the UHCD extract

Cross-tabulations between all three fields are produced to quantify agreement, discordances, and missing data patterns.

### 3. Disposition harmonisation
A rule-based function `harmonize_outcome()` creates a unified `disposition` variable with four modalities:
- **Retour domicile** — discharged home
- **UHCD puis RAD** — short observation then home discharge
- **Hospitalisation/transfert** — hospitalised or transferred
- **UHCD puis hospitalisation/transfert** — short observation then hospitalisation

The function prioritises `decision_urgence` (the more reliable administrative field), uses `disposition_med` to fill remaining NaNs, and applies the UHCD flag to further refine the category. A second pass `fill_nan_disposition()` resolves residual missingness using `disposition_med` alone.

### 4. Investigation of residual NaNs
Patients with an unresolvable `disposition` are characterised by triage score, discharge mode (`mode_sortie_chu`), and their cross-tabulation pattern, to understand which records are structurally uninformative.

### 5. Export
The final dataset (2022–2023 only) is saved as `df_adm_pv_ioa_med_radio_labo_uhcd_tabular.csv`.

In [30]:
import pandas as pd
import numpy as np

# --- Load ---
df_uhcd = pd.read_csv("../Datanad/subset_data/nda_uhcd_2022_2023.csv", dtype={'nda': str})
df_full = pd.read_csv("Datasets/df_adm_pv_ioa_med_radio_labo_tabular.csv", dtype={'nda': str})

# --- Clean NDA ---
df_uhcd['nda'] = df_uhcd['nda'].astype(str).str.strip()
df_full['nda'] = df_full['nda'].astype(str).str.strip()

# --- Filter 2022 and 2023 only ---
df_full['datetime_admission'] = pd.to_datetime(df_full['datetime_admission'], errors='coerce')
df_2223 = df_full[df_full['datetime_admission'].dt.year.isin([2022, 2023])].copy()
print(f"Patients 2022 and 2023 : {len(df_2223):,}")

# --- Merge ---
df_merged = pd.merge(df_2223, df_uhcd, on='nda', how='left')
print(f"after merge : {len(df_merged):,}")

# --- Comparaison ---
print("\n=== available columns for comparaison ===")
for col in ['decision_urgence', 'disposition_med', 'UHCD']:
    present = col in df_merged.columns
    print(f"  {col:20} : {'✅' if present else '❌ missing'}")

/tmp/ipykernel_2837416/2495157564.py:6: DtypeWarning: Columns (12,77,78,81,83,84,85,87,88,89,90,91,92,96,98,100,101,103,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,121,122,125,129,131,132,133,134,135,136,137,140,141,142,143,144,145) have mixed types. Specify dtype option on import or set low_memory=False.
  df_full = pd.read_csv("Datasets/df_adm_pv_ioa_med_radio_labo_tabular.csv", dtype={'nda': str})


Patients 2022 and 2023 : 57,389
after merge : 57,389

=== available columns for comparaison ===
  decision_urgence     : ✅
  disposition_med      : ✅
  UHCD                 : ✅


In [31]:
print(df_full['triage'].value_counts(dropna=False).to_string())
print(df_merged['triage'].value_counts(dropna=False).to_string())

triage
3.0    21072
4.0    16354
2.0    15941
5.0     3817
1.0      200
NaN        5
triage
3.0    21072
4.0    16354
2.0    15941
5.0     3817
1.0      200
NaN        5


In [32]:
# --- Comparison decision_urgence vs disposition_med vs UHCD ---

print(f"Total merged patients 2022 & 2023 : {len(df_merged):,}")

print("\n=== decision_urgence ===")
print(df_merged['decision_urgence'].value_counts(dropna=False).to_string())

print("\n=== disposition_med ===")
print(df_merged['disposition_med'].value_counts(dropna=False).to_string())

print("\n=== UHCD ===")
print(df_merged['UHCD'].value_counts(dropna=False).to_string())

print("\n=== CROSSING decision_urgence × UHCD ===")
print(pd.crosstab(
    df_merged['decision_urgence'].fillna('NaN'),
    df_merged['UHCD'].fillna('NaN'),
    margins=True
).to_string())

print("\n=== CROSSING disposition_med × UHCD ===")
print(pd.crosstab(
    df_merged['disposition_med'].fillna('NaN'),
    df_merged['UHCD'].fillna('NaN'),
    margins=True
).to_string())

print("\n=== CROSSING disposition_med × decision_urgence ===")
print(pd.crosstab(
    df_merged['disposition_med'].fillna('NaN'),
    df_merged['decision_urgence'].fillna('NaN'),
    margins=True
).to_string())

Total merged patients 2022 & 2023 : 57,389

=== decision_urgence ===
decision_urgence
Hospitalisation    30728
Consultation       25309
NaN                 1348
Pas de décision        4

=== disposition_med ===
disposition_med
Retour domicile                     29398
NaN                                 20168
Hospitalisation au CHU               4272
UHCD puis hospitalisation au CHU     1260
UHCD puis RAD                        1117
Transfert hors CHU                    888
UHCD puis transfert hors CHU          286

=== UHCD ===
UHCD
False    46001
True     11233
NaN        155

=== CROSSING decision_urgence × UHCD ===
UHCD              False   True  NaN    All
decision_urgence                          
Consultation      24648    660    1  25309
Hospitalisation   20318  10261  149  30728
NaN                1034    309    5   1348
Pas de décision       1      3    0      4
All               46001  11233  155  57389

=== CROSSING disposition_med × UHCD ===
UHCD                           

In [33]:
def harmonize_outcome(row):
    dec = str(row['decision_urgence']).strip() if pd.notna(row['decision_urgence']) else 'NaN'
    dis = str(row['disposition_med']).strip() if pd.notna(row['disposition_med']) else 'NaN'
    uhcd = bool(row['UHCD']) if pd.notna(row['UHCD']) else False

    # NaN total
    if dec == 'NaN' and dis == 'NaN':
        return np.nan

    # --- CONSULTATION ---
    if dec == 'Consultation':
        if dis in [
            'Hospitalisation au CHU',
            'UHCD puis hospitalisation au CHU',
            'NaN',
            'Retour domicile',
            'Transfert hors CHU',
            'UHCD puis RAD',
            'UHCD puis transfert hors CHU']:
            return 'UHCD puis RAD' if uhcd else 'Retour domicile'
        # if dis in ['Transfert hors CHU']:
        #     return 'UHCD puis transfert hors CHU' if uhcd else 'Transfert hors CHU'
        # if dis in ['Hospitalisation au CHU', 'NaN']:
        #     return 'UHCD puis RAD' if uhcd else 'Retour domicile'
        # if dis == 'Retour domicile':
        #     return 'UHCD puis RAD' if uhcd else 'Retour domicile'
        # if dis == 'UHCD puis RAD':
        #     return 'UHCD puis RAD' if uhcd else 'Retour domicile'
        # if dis == 'UHCD puis hospitalisation au CHU':
        #     return 'UHCD puis RAD' if uhcd else 'Retour domicile'
        # if dis in ['UHCD puis transfert hors CHU']:
        #     return 'UHCD puis RAD' if uhcd else 'Retour domicile'

    # --- HOSPITALISATION ---
    if dec == 'Hospitalisation':
        # if dis == 'Transfert hors CHU':
        #     return 'UHCD puis transfert hors CHU' if uhcd else 'Transfert hors CHU'
        if dis in [
            'Hospitalisation au CHU',
            'UHCD puis hospitalisation au CHU',
            'NaN',
            'Retour domicile',
            'Transfert hors CHU',
            'UHCD puis RAD',
            'UHCD puis transfert hors CHU']:
            return 'UHCD puis hospitalisation/transfert' if uhcd else 'Hospitalisation/transfert'
        #if dis == 'UHCD puis RAD':
        #    return 'UHCD puis hospitalisation/transfert' if uhcd else 'hospitalisation/transfert'
        #if dis == 'UHCD puis hospitalisation au CHU':
        #    return 'UHCD puis hospitalisation/transfert' if uhcd else 'Hospitalisation/transfert'
        #if dis == 'UHCD puis transfert hors CHU':
        #    return 'UHCD puis transfert hors CHU' if uhcd else 'Transfert hors CHU'

    return np.nan

df_merged['disposition'] = df_merged.apply(harmonize_outcome, axis=1)

print("=== OUTCOME DISTRIBUTION ===")
print(df_merged['disposition'].value_counts(dropna=False).to_string())

print("\n=== VERIFICATION : disposition × obs ===")
print(pd.crosstab(
    df_merged['disposition'].fillna('NaN'),
    df_merged['UHCD'].fillna('NaN'),
    margins=True
).to_string())

=== OUTCOME DISTRIBUTION ===
disposition
Retour domicile                        24649
Hospitalisation/transfert              20467
UHCD puis hospitalisation/transfert    10261
NaN                                     1352
UHCD puis RAD                            660

=== VERIFICATION : disposition × obs ===
UHCD                                 False   True  NaN    All
disposition                                                  
Hospitalisation/transfert            20318      0  149  20467
NaN                                   1035    312    5   1352
Retour domicile                      24648      0    1  24649
UHCD puis RAD                            0    660    0    660
UHCD puis hospitalisation/transfert      0  10261    0  10261
All                                  46001  11233  155  57389


In [34]:
# def harmonize_disposition(row):
#     dec  = str(row['decision_urgence']).strip() if pd.notna(row['decision_urgence']) else 'NaN'
#     uhcd = bool(row['UHCD']) if pd.notna(row['UHCD']) else False
#
#     if dec == 'Hospitalisation':
#         return 'UHCD puis Hospitalisation' if uhcd else 'Hospitalisation'
#     if dec == 'Consultation':
#         return 'UHCD puis RAD' if uhcd else 'Retour domicile'
#     if dec == 'Hospitalisation':
#         return 'UHCD puis transfert hors CHU' if uhcd else ''
#
#     return np.nan
#
# df_merged['disposition'] = df_merged.apply(harmonize_disposition, axis=1)
#
# print("=== disposition DISTRIBUTION ===")
# print(df_merged['disposition'].value_counts(dropna=False).to_string())

voir si les transferts je les considere comme des hospit ou pas

In [35]:
def fill_nan_disposition(row):
    if pd.notna(row['disposition']):
        return row['disposition']

    dis = str(row['disposition_med']).strip() if pd.notna(row['disposition_med']) else 'NaN'
    uhcd = bool(row['UHCD']) if pd.notna(row['UHCD']) else False


    if dis in [
        'Hospitalisation au CHU',
        'Transfert hors CHU',
        'UHCD puis hospitalisation au CHU',
        'UHCD puis transfert hors CHU']:
        return 'UHCD puis hospitalisation/transfert' if uhcd else 'Hospitalisation/transfert'

    if dis in ['Retour domicile', 'UHCD puis RAD']:
        return 'UHCD puis RAD' if uhcd else 'Retour domicile'



    return np.nan

df_merged['disposition'] = df_merged.apply(fill_nan_disposition, axis=1)

print("=== disposition DISTRIBUTION ===")
print(df_merged['disposition'].value_counts(dropna=False).to_string())

=== disposition DISTRIBUTION ===
disposition
Retour domicile                        25174
Hospitalisation/transfert              20553
UHCD puis hospitalisation/transfert    10319
UHCD puis RAD                            795
NaN                                      548


In [36]:
print("=== CROISEMENT disposition × disposition_med ===")
print(pd.crosstab(
    df_merged['disposition'].fillna('NaN'),
    df_merged['disposition_med'].fillna('NaN'),
    margins=True
).to_string())

=== CROISEMENT disposition × disposition_med ===
disposition_med                      Hospitalisation au CHU    NaN  Retour domicile  Transfert hors CHU  UHCD puis RAD  UHCD puis hospitalisation au CHU  UHCD puis transfert hors CHU    All
disposition                                                                                                                                                                                  
Hospitalisation/transfert                              2966   6646            10405                 410             65                                37                            24  20553
NaN                                                       0    548                0                   0              0                                 0                             0    548
Retour domicile                                          82   9102            15860                  96             30                                 2                             2  25174
U

In [37]:
# Qui sont les 548 patients avec disposition encore NaN ?
df_nan_disp = df_merged[df_merged['disposition'].isna()].copy()

print(f"Patients avec disposition NaN restants : {len(df_nan_disp):,}")

print("\n=== Score de tri ===")
print(df_nan_disp['triage'].value_counts(dropna=False).to_string())

print("\n=== decision_urgence ===")
print(df_nan_disp['decision_urgence'].value_counts(dropna=False).to_string())

print("\n=== disposition_med ===")
print(df_nan_disp['disposition_med'].value_counts(dropna=False).to_string())

print("\n=== UHCD ===")
print(df_nan_disp['UHCD'].value_counts(dropna=False).to_string())

print("\n=== Croisement decision_urgence x UHCD ===")
print(pd.crosstab(
    df_nan_disp['decision_urgence'].fillna('NaN'),
    df_nan_disp['UHCD'].fillna('NaN'),
    margins=True
).to_string())

Patients avec disposition NaN restants : 548

=== Score de tri ===
triage
3.0    201
4.0    171
2.0    142
5.0     33
1.0      1

=== decision_urgence ===
decision_urgence
NaN                546
Pas de décision      2

=== disposition_med ===
disposition_med
NaN    548

=== UHCD ===
UHCD
False    426
True     119
NaN        3

=== Croisement decision_urgence x UHCD ===
UHCD              False  True  NaN  All
decision_urgence                       
NaN                 426   117    3  546
Pas de décision       0     2    0    2
All                 426   119    3  548


In [38]:
print("=== mode_sortie_chu distribution for NA disposition ===\n")
print(df_merged[df_merged['disposition'].isna()]['mode_sortie_chu'].value_counts(dropna=False).to_string())

=== mode_sortie_chu distribution for NA disposition ===

mode_sortie_chu
A - Domicile                            314
NaN                                     200
B4 - Transfert PSY                       11
B1 - Transfert MCO                        8
C0 - Décédé                               7
B2 - Transfert SSR                        6
C1 - Transfert provisoire (<48h) MCO      2


In [39]:
# --- Merging in big dataset ---
# --- sure to keeep only 2022 and 2023
df_merged = df_merged[df_merged['datetime_admission'].dt.year.isin([2022, 2023])].copy()


print(f"Patients 2022 & 2023 only : {len(df_merged):,}")
print(f"Columns : {df_merged.columns.tolist()}")

# --- Export ---
df_merged.to_csv("Datasets/df_adm_pv_ioa_med_radio_labo_uhcd_tabular.csv", index=False)
print("✅ Saved.")

Patients 2022 & 2023 only : 57,389
Columns : ['nda', 'sex', 'age', 'uam_service', 'hospital', 'datetime_admission', 'date_sortie_urg', 'date_sortie_chu', 'date_sortie_urg_completee', 'mode_sortie_chu', 'decision_urgence', 'transport_grouped', 'date_hour_triage_begin', 'date_hour_triage_end', 'duration_triage_ioa_min', 'chief_complaint', 'triage', 'triage_raw', 'sbp', 'dbp', 'mbp', 'bp_status', 'is_bp_measured', 'hr', 'hr_status', 'is_hr_measured', 'temp', 'temp_status', 'is_temp_measured', 'sat', 'sat_status', 'is_sat_measured', 'rr', 'rr_status', 'is_rr_measured', 'o2_flow', 'o2_flow_status', 'is_o2_measured', 'gcs', 'gcs_status', 'is_gcs_measured', 'cap_blood_sugar_mmol_L', 'cap_blood_sugar_status', 'is_cap_blood_sugar_mmol_L_measured', 'pupil_right', 'pupil_left', 'pupils_status', 'anisocoria_status', 'is_pupils_measured', 'urine_dipstick_clean', 'is_urine_dipstick_clean_measured', 'urine_dipstick_clean_status', 'pain', 'pain_status', 'is_pain_measured', 'breathalyzer', 'breathalyze

# II. explo en attendant d'avoir toute les data

In [40]:
# import pandas as pd
#
# df = pd.read_csv("Datasets/df_adm_pv_ioa_med_radio_labo_tabular.csv", dtype={'nda': str})
#
# print("=== DISPOSITION_MED DISTRIBUTION ===\n")
#
# n_total = len(df)
#
# # Value counts
# vc = df['disposition_med'].value_counts(dropna=False)
# print(f"{'Value':<30} {'N':>8} {'%':>8}")
# print("-" * 48)
# for val, count in vc.items():
#     label = str(val) if not pd.isna(val) else "NA"
#     print(f"{label:<30} {count:>8} {100*count/n_total:>7.1f}%")
#
# print(f"\nTotal: {n_total:,}")
# print(f"Missing: {df['disposition_med'].isna().sum():,} ({100*df['disposition_med'].isna().mean():.1f}%)")
# print(f"Unique non-null values: {df['disposition_med'].nunique()}")
#
